# Lab 1 - Microstates of a system of coins (Person A: questions 1a, 1b)

Runs the code in `src/microstates.py`, its tests and `main.py`. Use **Run All** in VS Code with the `statphys_lab1` kernel.

## Setup
Adds the repo root to the path so `src` can be imported from `notebooks/`.

In [ ]:
import subprocess
import sys
from math import comb
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.microstates import (
    benchmark_enumeration,
    count_heads,
    list_microstates_by_macrostate,
    macrostate_histogram,
    macrostate_histogram_fast,
    macrostate_label,
    microstates,
    state_to_string,
)

print("Repo root:", ROOT)

## Unit tests
Runs `pytest` on `tests/` (same as typing `pytest` in the terminal).

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-v"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout[-3000:])

## How the bitwise encoding works
Each integer `s` in $[0, 2^N)$ is one microstate: bit $i$ is coin $i$ (1 = heads). The number of heads $k$ is the number of 1-bits of `s`.

In [ ]:
N = 3
for s in microstates(N):
    print(f"s = {s}  binary = {s:0{N}b}  coins = {state_to_string(s, N)}  heads = {count_heads(s)}")

## 1a: all $2^7 = 128$ microstates of $N = 7$ coins

In [ ]:
N = 7
groups = list_microstates_by_macrostate(N)
print(f"{'macrostate':<22}{'# microstates':>14}")
for k, states in groups.items():
    print(f"{macrostate_label(k, N):<22}{len(states):>14}")
print(f"{'total':<22}{sum(len(s) for s in groups.values()):>14}")

In [ ]:
for k, states in groups.items():
    print(f"{macrostate_label(k, N)} ({len(states)}): {', '.join(states)}\n")

## Check: the enumeration gives the binomial coefficients $\Omega(N,k) = \binom{N}{k}$

In [ ]:
for N in [5, 7, 10, 16]:
    omega = macrostate_histogram(N)
    print(f"N = {N:2d}: {omega}  matches comb: {omega == [comb(N, k) for k in range(N + 1)]}  sum = 2^{N}: {sum(omega) == 2**N}")

## 1b: pure-Python loop vs numpy (chunked) version

In [ ]:
for N in [16, 18, 20]:
    t_py, s_py, _ = benchmark_enumeration(N, repeats=3, fast=False)
    t_np, s_np, _ = benchmark_enumeration(N, repeats=3, fast=True)
    print(f"N = {N}: Python {t_py:.3f} +/- {s_py:.3f} s, numpy {t_np:.4f} +/- {s_np:.4f} s, speed-up x{t_py / t_np:.0f}")

## 1b: enumeration time and memory vs $N$
Each run is repeated 3 times (mean $\pm$ std). Peak RAM measured with `tracemalloc`.

In [ ]:
ns, means, stds, rams = [], [], [], []
print(f"{'N':>3}{'2^N':>14}{'time (s)':>22}{'peak RAM (MiB)':>16}")
for N in range(10, 27, 2):
    mean, std, ram = benchmark_enumeration(N, repeats=3)
    ns.append(N); means.append(mean); stds.append(std); rams.append(ram)
    print(f"{N:>3}{2**N:>14}{f'{mean:.4f} +/- {std:.4f}':>22}{ram:>16.2f}")

In [ ]:
ns_arr, means_arr = np.array(ns), np.array(means)
fit = ns_arr >= 16
b, a = np.polyfit(ns_arr[fit], np.log(means_arr[fit]), 1)
n_limit = (np.log(300) - a) / b
print(f"slope b = {b:.3f} (theory ln 2 = {np.log(2):.3f})")
print(f"extrapolated largest N in 5 min: {n_limit:.1f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.errorbar(ns, means, yerr=stds, fmt="o", capsize=3, label="measured (mean +/- std)")
n_line = np.linspace(min(ns), n_limit, 100)
ax1.plot(n_line, np.exp(a + b * n_line), "--", label=f"fit, slope {b:.3f}")
ax1.axhline(300, color="gray", linestyle=":", label="5 min limit")
ax1.set_yscale("log"); ax1.set_xlabel("Number of coins $N$"); ax1.set_ylabel("Time (s)"); ax1.legend()
ax2.plot(ns, rams, "s-")
ax2.set_xlabel("Number of coins $N$"); ax2.set_ylabel("Peak RAM (MiB)")
ax2.set_title("Memory stops growing once N exceeds the chunk size (2^16)")
fig.tight_layout()
plt.show()

## Run `main.py`
Same as `python main.py` in the terminal: prints 1a and 1b and writes the figure to `report/figures/`.

In [ ]:
result = subprocess.run([sys.executable, "main.py"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout, result.stderr)

## Optional: measure the real 5-minute limit
Slow (several minutes). Change `False` to `True` to run it.

In [ ]:
RUN_FIND_LIMIT = False
if RUN_FIND_LIMIT:
    result = subprocess.run([sys.executable, "main.py", "--find-limit"], cwd=ROOT, capture_output=True, text=True)
    print(result.stdout, result.stderr)